[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet1520/langgraph-workshop/blob/main/notebooks/level-1-basics/10_chatbot_summarization_stream.ipynb)

# Chatbot: Summarization & Streaming

**Level 1** · Part of the [LangGraph Workshop](https://github.com/Abhijeet1520/langgraph-workshop).

> **Source:** migrated from `snehotosh/langgraph-notebooks` ([original](https://github.com/snehotosh/langgraph-notebooks/blob/main/8_build_chatbot_summarization_and_stream.ipynb)) — see [PROVENANCE.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/PROVENANCE.md).

**Provider setup:** set `OPENAI_API_KEY` / `OPENAI_API_BASE` / `OPENAI_MODEL` first — [docs/setup/providers.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/setup/providers.md). The first cells install requirements and verify the connection.

## Goal

This module will dive into Streaming, and building on memory.

### Learning Steps:
- `Streaming`

In [ ]:
# --- Workshop provider config (edit ONLY these 3 values) ---------------------
import os

os.environ.setdefault("OPENAI_API_KEY", "")   # <- paste your key
os.environ.setdefault("OPENAI_API_BASE", "")  # <- paste your base URL
os.environ.setdefault("OPENAI_MODEL", "openai/gpt-4.1-mini")  # <- your model id

# Verify
for _v in ("OPENAI_API_KEY", "OPENAI_API_BASE", "OPENAI_MODEL"):
    print(_v, "=", "set" if os.getenv(_v) else "**MISSING — see docs/setup/providers.md**")
# -----------------------------------------------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
import os

def make_llm(**overrides):
    """Provider-agnostic LLM factory used across the workshop."""
    cfg = dict(
        model=os.environ["OPENAI_MODEL"],
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE"],
        temperature=0,
    )
    cfg.update(overrides)
    return ChatOpenAI(**cfg)

llm = make_llm()

# Connection check (1 token)
resp = llm.invoke("Say OK")
print("✅ LLM reachable:", resp.content[:60])


#### Standard imports

In [ ]:
from typing import TypedDict, Annotated, Literal

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, RemoveMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from langgraph.graph import START, END, StateGraph, add_messages, MessagesState, MessageGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from IPython.display import display, Image

In [ ]:
load_dotenv()

In [ ]:
llm = make_llm()

In [ ]:
'''
MessageState by default give messages as a key and adds "add_messages" automatically and includes TypedDict
So, its a dictionary of list
e.g.
messages: list[AnyMessages]
summary: str
'''
class State(MessagesState):
    summary: str

In [ ]:
def call_model(state: State):
    summary = state.get("summary","")

    if summary:
        system_message =  f"Summary of conversation earlier: {summary}"
        messages = [SystemMessage(content=system_message)] + state["messages"]
    else:
        messages = state["messages"]

    response = llm.invoke(messages)
    return {"messages": [response]}

In [ ]:
def summarize_conversation(state: State):
    
    # First, we get any existing summary
    summary = state.get("summary", "")

    # Create our summarization prompt 
    if summary:
        
        # A summary already exists
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
        
    else:
        summary_message = "Create a summary of the conversation above:"

    # Add prompt to our history
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = llm.invoke(messages)
    
    # Delete all but the 2 most recent messages
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]] ## its a list
    return {"summary": response.content, "messages": delete_messages}

In [ ]:
# Determine whether to end or summarize the conversation
def should_continue(state: State) -> Literal ["summarize_conversation",END]:
    
    """Return the next node to execute."""
    
    messages = state["messages"]
    
    # If there are more than six messages, then we summarize the conversation
    if len(messages) > 4:
        return "summarize_conversation"
    
    # Otherwise we can just end
    return END

In [ ]:
# Define a new graph
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node("summarize_conversation", summarize_conversation)

# Set the entrypoint as conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# Compile
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "1"}}

# Start conversation
input_message = HumanMessage(content="hi! I'm Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

if "summary" in output:
    print("\n--- Updated Summary ---")
    print(output["summary"])

input_message = HumanMessage(content="what's my name?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

if "summary" in output:
    print("\n--- Updated Summary ---")
    print(output["summary"])

input_message = HumanMessage(content="i like the 49ers!")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

if "summary" in output:
    print("\n--- Updated Summary ---")
    print(output["summary"])

In [ ]:
input_message = HumanMessage(content="i like Nick Bosa, isn't he the highest paid defensive player?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

if "summary" in output:
    print("\n--- Updated Summary ---")
    print(output["summary"])

In [ ]:
graph.get_state(config).values.get("summary","")

## Streaming
- `graph.stream()` yields partial state updates as a generator.
- `stream_mode="updates"`: LangGraph yields incremental state updates — i.e., only the parts of the graph state that have changed since the previous chunk.
-  `stream_mode="values"`: LangGraph yields the entire state value after each node completes, not just the updates.

In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "1"}}

# Start conversation
for chunk in graph.stream({"messages": [HumanMessage(content="hi! I'm Lance")]}, config, stream_mode="updates"):
    print(chunk)

In [ ]:
# Start conversation
for chunk in graph.stream(
    {"messages": [HumanMessage(content="hi! I'm Lance")]}, 
    config, 
    stream_mode="updates"
):
    for m in chunk['conversation']["messages"]:
        m.pretty_print()

#### `astream_events()`

Normally, `graph.invoke()` or `graph.stream()` only give you final states (like “messages,” “summary,” etc.) or partial state updates between nodes.

But under the hood, there’s much more happening:
- Each node in your LangGraph may invoke an LLM call,
- The LLM call itself streams tokens as they’re generated,
- LangGraph tracks internal node start/end events, message updates, and errors.

👉 `graph.astream_events()` lets you tap into that entire event stream, so you can observe what’s happening inside the graph, in real time, token by token or node by node.

| Use Case                      | Benefit                                                                         |
| ----------------------------- | ------------------------------------------------------------------------------- |
| **Real-time token streaming** | You can print each token as the LLM generates it (like OpenAI’s streaming API). |
| **Tracing / Debugging**       | You can see when each node starts and ends, and what data it emits.             |
| **Building interactive UIs**  | Perfect for updating chat bubbles, progress indicators, or dashboards live.     |
| **Profiling**                 | You can measure latency between node starts/ends.                               |


| Event Type             | Description                              | Common Use                                  |
| ---------------------- | ---------------------------------------- | ------------------------------------------- |
| `on_chat_model_start`  | When a Chat Model node (LLM call) begins | Start timing, allocate log record           |
| `on_chat_model_stream` | When partial tokens are streamed back    | Real-time token streaming (✅ the juicy one) |
| `on_chat_model_end`    | When the LLM finishes                    | Close stream, compute latency               |
| `on_tool_start`        | When a tool/function node begins         | Debug tool call latency                     |
| `on_tool_end`          | When tool execution completes            | Get result, log                             |
| `on_node_end`          | When a graph node finishes               | Update UI or save intermediate state        |
| `on_graph_end`         | When the entire graph finishes           | Final output ready                          |


In [ ]:
config = {"configurable": {"thread_id": "1"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")

async for event in graph.astream_events(
    {"messages": [input_message]}, 
    config, 
    version="v2"
):
    print(f"Node: {event['metadata'].get('langgraph_node','')}. Type: {event['event']}. Name: {event['name']}")

In [ ]:
node_to_stream = 'conversation'
config = {"configurable": {"thread_id": "1"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # Get chat model tokens from a particular node 
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        print(event["data"])

In [ ]:
config = {"configurable": {"thread_id": "1"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # Get chat model tokens from a particular node 
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        data = event["data"]
        print(data["chunk"].content, end="|")